In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_percentage_error
)

In [2]:
DATA_DIR = Path("./data_versions")

version_files = sorted(
    list(DATA_DIR.glob("v*.csv.gz")) +
    list(DATA_DIR.glob("v*.csv"))
)

print(f"Found {len(version_files)} dataset versions:")

for file_path in version_files:
    print(file_path.name)

Found 8 dataset versions:
v0_week5_baseline.csv.gz
v1_school_district.csv.gz
v2_hoa.csv.gz
v3_ratio.csv.gz
v4_school_hoa.csv.gz
v5_school_ratio.csv.gz
v6_all_features.csv.gz
version_summary.csv


In [ ]:
RANDOM_STATE = 420

candidate_models = [
    {
        "Model": "Decision Tree",
        "Estimator": DecisionTreeRegressor(
            max_depth=18,
            min_samples_leaf=10,
            random_state=RANDOM_STATE
        )
    },
    {
        "Model": "Random Forest",
        "Estimator": RandomForestRegressor(
            n_estimators=60,
            max_depth=30,
            min_samples_leaf=10,
            max_features=0.7,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    }
]

In [4]:
def evaluate_additional_model(file_path, model_name, estimator):
    """
    Load one feature-set version, preprocess its features,
    train one additional model, and return evaluation metrics.
    """

    # Read one dataset version
    df = pd.read_csv(file_path)

    # Create a clean version name
    version_name = file_path.name

    if version_name.endswith(".csv.gz"):
        version_name = version_name[:-7]
    elif version_name.endswith(".csv"):
        version_name = version_name[:-4]

    # Check required columns
    required_columns = {"ClosePrice", "Dataset"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    # Replace infinite values with NaN
    df = df.replace([np.inf, -np.inf], np.nan)

    # Keep only Train and Test rows
    df = df[df["Dataset"].isin(["Train", "Test"])].copy()

    train_df = df[df["Dataset"] == "Train"].copy()
    test_df = df[df["Dataset"] == "Test"].copy()

    if train_df.empty:
        raise ValueError("The training dataset is empty.")

    if test_df.empty:
        raise ValueError("The test dataset is empty.")

    # Remove rows with invalid target values
    train_df = train_df[
        train_df["ClosePrice"].notna()
        & np.isfinite(train_df["ClosePrice"])
        & (train_df["ClosePrice"] > 0)
    ].copy()

    test_df = test_df[
        test_df["ClosePrice"].notna()
        & np.isfinite(test_df["ClosePrice"])
        & (test_df["ClosePrice"] > 0)
    ].copy()

    if train_df.empty or test_df.empty:
        raise ValueError(
            "No valid target rows remain after filtering ClosePrice."
        )

    # Targets
    y_train = train_df["ClosePrice"].to_numpy(dtype=np.float64)
    y_test = test_df["ClosePrice"].to_numpy(dtype=np.float64)

    # Drop non-predictor columns
    columns_to_drop = [
        "ClosePrice",
        "Dataset",
        "CloseDate"
    ]

    X_train = train_df.drop(
        columns=columns_to_drop,
        errors="ignore"
    ).copy()

    X_test = test_df.drop(
        columns=columns_to_drop,
        errors="ignore"
    ).copy()

    # Align test columns with training columns
    X_test = X_test.reindex(columns=X_train.columns)

    # Drop columns that are completely missing in training
    all_missing_columns = X_train.columns[
        X_train.isna().all()
    ].tolist()

    if all_missing_columns:
        X_train = X_train.drop(columns=all_missing_columns)

        X_test = X_test.drop(
            columns=all_missing_columns,
            errors="ignore"
        )

    if X_train.shape[1] == 0:
        raise ValueError("No usable predictor columns remain.")

    feature_columns = X_train.columns.tolist()

    # Detect numeric and categorical columns
    numeric_columns = X_train.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_columns = X_train.select_dtypes(
        exclude=["number", "bool"]
    ).columns.tolist()

    # Convert boolean columns to integer
    for column in numeric_columns:
        if X_train[column].dtype == bool:
            X_train[column] = X_train[column].astype(int)
            X_test[column] = X_test[column].astype(int)

    # Numeric preprocessing
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            )
        ]
    )

    # Categorical preprocessing
    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "one_hot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="first"
                )
            )
        ]
    )

    transformers = []

    if numeric_columns:
        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_columns
            )
        )

    if categorical_columns:
        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns
            )
        )

    if not transformers:
        raise ValueError(
            "No numeric or categorical predictors were found."
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    model_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator)
        ]
    )

    # Fit model
    model_pipeline.fit(X_train, y_train)

    # Predictions
    train_pred = model_pipeline.predict(X_train)
    test_pred = model_pipeline.predict(X_test)

    # Absolute percentage error
    train_ape = np.abs(
        (y_train - train_pred) / y_train
    )

    test_ape = np.abs(
        (y_test - test_pred) / y_test
    )

    # Count features after preprocessing
    encoded_feature_count = (
        model_pipeline
        .named_steps["preprocessor"]
        .transform(X_train.iloc[:1])
        .shape[1]
    )

    return {
        "Version": version_name,
        "Model": model_name,
        "Rows": len(df),
        "Train Rows": len(train_df),
        "Test Rows": len(test_df),
        "Features Before Encoding": len(feature_columns),
        "Numeric Features": len(numeric_columns),
        "Categorical Features": len(categorical_columns),
        "Encoded Features": encoded_feature_count,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
        "Train RMSE": mean_squared_error(
            y_train,
            train_pred
        ) ** 0.5,
        "Test RMSE": mean_squared_error(
            y_test,
            test_pred
        ) ** 0.5,
        "Train MAPE": mean_absolute_percentage_error(
            y_train,
            train_pred
        ),
        "Test MAPE": mean_absolute_percentage_error(
            y_test,
            test_pred
        ),
        "Train MdAPE": np.median(train_ape),
        "Test MdAPE": np.median(test_ape)
    }

In [5]:
additional_results = []
failed_models = []

for file_path in version_files:

    for model_spec in candidate_models:

        model_name = model_spec["Model"]
        estimator = model_spec["Estimator"]

        print("=" * 80)
        print(
            f"Running {model_name}: "
            f"{file_path.name}"
        )

        try:
            result = evaluate_additional_model(
                file_path=file_path,
                model_name=model_name,
                estimator=estimator
            )

            additional_results.append(result)

            print(
                "Completed | "
                f"Test R2: {result['Test R2']:.4f} | "
                f"Test RMSE: ${result['Test RMSE']:,.0f} | "
                f"Test MAPE: {result['Test MAPE']:.2%} | "
                f"Test MdAPE: {result['Test MdAPE']:.2%}"
            )

        except Exception as error:
            failed_models.append(
                {
                    "Version": file_path.name,
                    "Model": model_name,
                    "Error": str(error)
                }
            )

            print(
                f"Failed: {model_name} | "
                f"{file_path.name}"
            )
            print(f"Reason: {error}")

Running Decision Tree: v0_week5_baseline.csv.gz
Completed | Test R2: 0.4655 | Test RMSE: $1,227,122 | Test MAPE: 22.60% | Test MdAPE: 11.85%
Running Random Forest: v0_week5_baseline.csv.gz
Completed | Test R2: 0.5031 | Test RMSE: $1,183,099 | Test MAPE: 20.24% | Test MdAPE: 10.10%
Running Decision Tree: v1_school_district.csv.gz


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Completed | Test R2: 0.4660 | Test RMSE: $1,226,491 | Test MAPE: 21.82% | Test MdAPE: 11.91%
Running Random Forest: v1_school_district.csv.gz


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Completed | Test R2: 0.5022 | Test RMSE: $1,184,239 | Test MAPE: 19.80% | Test MdAPE: 9.79%
Running Decision Tree: v2_hoa.csv.gz
Completed | Test R2: 0.4659 | Test RMSE: $1,226,598 | Test MAPE: 24.22% | Test MdAPE: 11.90%
Running Random Forest: v2_hoa.csv.gz
Completed | Test R2: 0.4997 | Test RMSE: $1,187,237 | Test MAPE: 20.12% | Test MdAPE: 9.93%
Running Decision Tree: v3_ratio.csv.gz
Completed | Test R2: 0.4649 | Test RMSE: $1,227,804 | Test MAPE: 22.63% | Test MdAPE: 11.85%
Running Random Forest: v3_ratio.csv.gz
Completed | Test R2: 0.4970 | Test RMSE: $1,190,445 | Test MAPE: 20.28% | Test MdAPE: 10.10%
Running Decision Tree: v4_school_hoa.csv.gz


...

Completed | Test R2: 0.5002 | Test RMSE: $1,186,604 | Test MAPE: 19.93% | Test MdAPE: 9.64%
Running Decision Tree: v6_all_features.csv.gz


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Completed | Test R2: 0.4648 | Test RMSE: $1,227,873 | Test MAPE: 23.07% | Test MdAPE: 11.92%
Running Random Forest: v6_all_features.csv.gz


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Completed | Test R2: 0.5034 | Test RMSE: $1,182,835 | Test MAPE: 19.69% | Test MdAPE: 9.65%
Running Decision Tree: version_summary.csv
Failed: Decision Tree | version_summary.csv
Reason: Missing required columns: ['ClosePrice', 'Dataset']
Running Random Forest: version_summary.csv
Failed: Random Forest | version_summary.csv
Reason: Missing required columns: ['ClosePrice', 'Dataset']


In [8]:
if additional_results:
    additional_results_df = pd.DataFrame(
        additional_results
    )

    additional_results_df = (
        additional_results_df
        .sort_values(
            by=["Test R2", "Test MdAPE"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

    display(
        additional_results_df[
            [
                "Version",
                "Model",
                "Features Before Encoding",
                "Encoded Features",
                "Train R2",
                "Test R2",
                "Test RMSE",
                "Test MAPE",
                "Test MdAPE"
            ]
        ]
    )
else:
    print("No additional model completed successfully.")

,Version,Model,Features Before Encoding,Encoded Features,Train R2,Test R2,Test RMSE,Test MAPE,Test MdAPE
0,v4_school_hoa,Random Forest,991,1665,0.830099,0.504751,1.181187e+06,0.194914,0.096613
1,v6_all_features,Random Forest,992,1666,0.831435,0.503368,1.182835e+06,0.196926,0.096540
2,v0_week5_baseline,Random Forest,987,987,0.825323,0.503147,1.183099e+06,0.202426,0.101011
3,v1_school_district,Random Forest,990,1664,0.829074,0.502189,1.184239e+06,0.198049,0.097895
4,v5_school_ratio,Random Forest,991,1665,0.827501,0.500199,1.186604e+06,0.199273,0.096375
5,v2_hoa,Random Forest,988,988,0.824835,0.499666,1.187237e+06,0.201188,0.099253
6,v3_ratio,Random Forest,988,988,0.821771,0.496958,1.190445e+06,0.202850,0.101022
7,v1_school_district,Decision Tree,990,1664,0.814860,0.466033,1.226491e+06,0.218164,0.119107
8,v2_hoa,Decision Tree,988,988,0.818780,0.465940,1.226598e+06,0.242249,0.118996
9,v5_school_ratio,Decision Tree,991,1665,0.815124,0.465896,1.226648e+06,0.218288,0.119232


In [9]:
best_by_model = (
    additional_results_df
    .sort_values(
        by=["Model", "Test R2"],
        ascending=[True, False]
    )
    .groupby("Model", as_index=False)
    .first()
)

display(
    best_by_model[
        [
            "Model",
            "Version",
            "Test R2",
            "Test RMSE",
            "Test MAPE",
            "Test MdAPE"
        ]
    ]
)

,Model,Version,Test R2,Test RMSE,Test MAPE,Test MdAPE
0,Decision Tree,v1_school_district,0.466033,1.226491e+06,0.218164,0.119107
1,Random Forest,v4_school_hoa,0.504751,1.181187e+06,0.194914,0.096613
